In [26]:
# pip install pandas sqlalchemy psycopg2-binary
import pandas as pd
from sqlalchemy import create_engine, text

# ---- Direct connection config (no env) ----
PG_USER = "postgres"
PG_PASSWORD = "tip_pwd"
PG_HOST = "localhost"
PG_PORT = "5432"
PG_DB = "tip"

SCHEMA = "gold"

# Build engine
engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    pool_pre_ping=True,
)

def list_relations(schema: str) -> tuple[list[str], list[str]]:
    """Return (tables, views) in a schema."""
    with engine.begin() as conn:
        tables = pd.read_sql(
            text("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = :s AND table_type='BASE TABLE'
                ORDER BY table_name
            """),
            conn,
            params={"s": schema},
        )["table_name"].tolist()

        views = pd.read_sql(
            text("""
                SELECT table_name
                FROM information_schema.views
                WHERE table_schema = :s
                ORDER BY table_name
            """),
            conn,
            params={"s": schema},
        )["table_name"].tolist()
    return tables, views

def load_schema(schema: str) -> dict[str, pd.DataFrame]:
    """
    Loads every table & view in `schema` into DataFrames.
    Returns a dict keyed by fully qualified name, e.g. 'gold.dim_cve'.
    """
    dfs: dict[str, pd.DataFrame] = {}
    tables, views = list_relations(schema)

    with engine.begin() as conn:
        # ensure search_path includes schema for simple SELECTs on views
        conn.execute(text(f"SET search_path TO {schema}, public"))

        # Tables
        for t in tables:
            fq = f"{schema}.{t}"
            dfs[fq] = pd.read_sql(text(f'SELECT * FROM "{schema}"."{t}"'), conn)

        # Views (optional: comment out if you only want base tables)
        for v in views:
            fq = f"{schema}.{v}"
            dfs[fq] = pd.read_sql(text(f'SELECT * FROM "{schema}"."{v}"'), conn)

    return dfs

if __name__ == "__main__":
    dfs = load_schema(SCHEMA)


In [27]:
print(dfs)

{'gold.bridge_cve_products':         bridge_id          cve_id  product_id                 created_at
0               1  CVE-2024-21732           1 2025-10-22 14:47:45.117729
1               2   CVE-2024-0181           2 2025-10-22 14:47:45.117729
2               3   CVE-2024-0182           3 2025-10-22 14:47:45.117729
3               4   CVE-2024-0183           2 2025-10-22 14:47:45.117729
4               5   CVE-2024-0184           2 2025-10-22 14:47:45.117729
...           ...             ...         ...                        ...
321088     321089  CVE-2002-20001        2973 2025-10-23 17:48:07.336689
321089     321090  CVE-2002-20001        2974 2025-10-23 17:48:07.336689
321090     321091  CVE-2002-20001        2975 2025-10-23 17:48:07.336689
321091     321092  CVE-2002-20001        2976 2025-10-23 17:48:07.336689
321092     321093  CVE-2002-20001        2977 2025-10-23 17:48:07.336689

[321093 rows x 4 columns], 'gold.cvss_v2':        cvss_v2_id          cve_id  source_id  cvss_

In [28]:
dim_cve.head()

,cve_id,vulnarbilit,published_date,last_modified,loaded_at,cve_year,remotely_exploit,source_identifier,created_at
0,CVE-2024-0001,uncategorized,2024-09-23 18:15:04.070,2024-09-27 14:08:57.327,2025-10-22 14:46:38.231217,2024,None,psirt@purestorage.com,2025-10-22 14:47:16.169678
1,CVE-2024-0002,authn_authz,2024-09-23 18:15:04.410,2024-09-27 14:13:24.427,2025-10-22 14:46:38.231217,2024,None,psirt@purestorage.com,2025-10-22 14:47:16.169678
2,CVE-2024-0003,authn_authz,2024-09-23 18:15:04.697,2024-09-27 14:23:58.243,2025-10-22 14:46:38.231217,2024,None,psirt@purestorage.com,2025-10-22 14:47:16.169678
3,CVE-2024-0004,injection,2024-09-23 18:15:04.973,2024-09-27 14:24:41.277,2025-10-22 14:46:38.231217,2024,None,psirt@purestorage.com,2025-10-22 14:47:16.169678
4,CVE-2024-0005,injection,2024-09-23 18:15:05.233,2024-09-27 15:25:40.980,2025-10-22 14:46:38.231217,2024,None,psirt@purestorage.com,2025-10-22 14:47:16.169678


In [29]:
cvss_v3.head()

,cvss_v3_id,cve_id,source_id,cvss_version,cvss_score,cvss_severity,cvss_vector,cvss_v3_base_av,cvss_v3_base_ac,cvss_v3_base_pr,cvss_v3_base_ui,cvss_v3_base_s,cvss_v3_base_c,cvss_v3_base_i,cvss_v3_base_a,cvss_exploitability_score,cvss_impact_score,created_at
0,1,CVE-2024-21732,139,CVSS 3.1,6.1,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,N,L,N,R,C,L,L,N,2.8,2.7,2025-10-22 14:47:26.711646
1,2,CVE-2024-21732,8,CVSS 3.1,6.1,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,N,L,N,R,C,L,L,N,2.8,2.7,2025-10-22 14:47:26.711646
2,3,CVE-2024-0181,81,CVSS 3.1,2.4,LOW,CVSS:3.1/AV:N/AC:L/PR:H/UI:R/S:U/C:N/I:L/A:N,N,L,H,R,U,N,L,N,0.9,1.4,2025-10-22 14:47:26.711646
3,4,CVE-2024-0181,139,CVSS 3.1,4.8,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:H/UI:R/S:C/C:L/I:L/A:N,N,L,H,R,C,L,L,N,1.7,2.7,2025-10-22 14:47:26.711646
4,5,CVE-2024-0182,81,CVSS 3.1,7.3,HIGH,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:L/A:L,N,L,N,N,U,L,L,L,3.9,3.4,2025-10-22 14:47:26.711646


In [30]:
# pip install pandas sqlalchemy psycopg2-binary
import pandas as pd
from sqlalchemy import create_engine, text

# --- Direct connection (no .env used) ---
PG_USER = "postgres"
PG_PASSWORD = "tip_pwd"
PG_HOST = "localhost"
PG_PORT = "5432"
PG_DB = "tip"

# Create SQLAlchemy engine
engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    pool_pre_ping=True
)

# --- Load raw.cve_details as a DataFrame ---
with engine.begin() as conn:
    df_cve_details = pd.read_sql(text('SELECT * FROM "raw"."cve_details"'), conn)

# --- Display preview ---
print("✅ Loaded rows:", len(df_cve_details))

✅ Loaded rows: 6770


In [31]:
df_cve_details["remotely_exploit"].unique()

array([None], dtype=object)

In [34]:
# pip install pandas sqlalchemy psycopg2-binary
import pandas as pd
from sqlalchemy import create_engine, text

# --- Direct connection (no .env used) ---
PG_USER = "postgres"
PG_PASSWORD = "tip_pwd"
PG_HOST = "localhost"
PG_PORT = "5432"
PG_DB = "tip"

# Create SQLAlchemy engine
engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    pool_pre_ping=True
)

# --- Load raw.cve_details as a DataFrame ---
with engine.begin() as conn:
    df_cve_details = pd.read_sql(text('SELECT * FROM "silver"."cve_cleaned"'), conn)

# --- Display preview ---
print("✅ Loaded rows:", len(df_cve_details))

✅ Loaded rows: 175392


In [35]:
df_cve_details.head()

,cve_id,vulnarbilit,published_date,last_modified,loaded_at,remotely_exploit,source_identifier,affected_products,cvss_scores,created_at,updated_at
0,CVE-1999-0095,uncategorized,1988-10-01 04:00:00,2025-04-03 01:03:51.193,2025-10-23 19:19:48.397750,None,cve@mitre.org,"[{""vendor"":""eric_allman"",""product"":""sendmail""}]","[{""type"":""Primary"",""score"":10.0,""vector"":""AV:N...",2025-10-23 19:21:37.209109,2025-10-23 19:21:37.209109
1,CVE-1999-0082,uncategorized,1988-11-11 05:00:00,2025-04-03 01:03:51.193,2025-10-23 19:19:48.397750,None,cve@mitre.org,"[{""vendor"":""ftp"",""product"":""ftp""},{""vendor"":""f...","[{""type"":""Primary"",""score"":10.0,""vector"":""AV:N...",2025-10-23 19:21:37.209109,2025-10-23 19:21:37.209109
2,CVE-1999-1471,uncategorized,1989-01-01 05:00:00,2025-04-03 01:03:51.193,2025-10-23 19:19:48.397750,None,cve@mitre.org,"[{""vendor"":""bsd"",""product"":""bsd""}]","[{""type"":""Primary"",""score"":7.2,""vector"":""AV:L/...",2025-10-23 19:21:37.209109,2025-10-23 19:21:37.209109
3,CVE-1999-1122,uncategorized,1989-07-26 04:00:00,2025-04-03 01:03:51.193,2025-10-23 19:19:48.397750,None,cve@mitre.org,"[{""vendor"":""sun"",""product"":""sunos""}]","[{""type"":""Primary"",""score"":4.6,""vector"":""AV:L/...",2025-10-23 19:21:37.209109,2025-10-23 19:21:37.209109
4,CVE-1999-1467,uncategorized,1989-10-26 04:00:00,2025-04-03 01:03:51.193,2025-10-23 19:19:48.397750,None,cve@mitre.org,"[{""vendor"":""sun"",""product"":""sunos""}]","[{""type"":""Primary"",""score"":10.0,""vector"":""AV:N...",2025-10-23 19:21:37.209109,2025-10-23 19:21:37.209109


In [37]:
df_cve_details["vulnarbilit"].unique()

array(['uncategorized', 'authn_authz', 'injection', 'memory_corruption',
       'input_validation', 'information_disclosure', 'config_permissions',
       'cryptography', 'race_condition', 'resource_management_dos', 'xss',
       'path_traversal', 'sql_injection', 'ssrf', 'deserialization',
       'open_redirect', 'xxe'], dtype=object)